# Establishing Connections to Databases

In [31]:
bridge_sfn = '0130192'

## TIMS 'doticsqlp31' Data Source

In [2]:
import pandas as pd
import sqlalchemy as db
from sqlalchemy import create_engine, MetaData, text, inspect
from sqlalchemy.engine import URL

In [3]:
connection_string = (
    r"Driver=ODBC Driver 17 for SQL Server;"
    r"Server=doticsqlp31;"
    r"Database=TIMS;"
    r"Trusted_Connection=yes;"
)

connection_url = URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})

engine = create_engine(connection_url)
conn = engine.connect()

## 'DOTWarehousePrd' Data Source

In [4]:
connection_string = (
    r"Driver=SQL Server;"
    r"Server=DOTWarehousePrd;"
    r"Database=Warehouse;"
    r"Trusted_Connection=yes;"
)
connection_url = URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})
engine = create_engine(connection_url)
conn = engine.connect()

In [5]:
# Get data from 1985-1990 table - Huge, don't do this.
# hist85_90DataQuery = text("SELECT * from BMS_1985_1990")

In [6]:
# hist_bridge_data = pd.read_sql(hist85_90DataQuery, conn)
# hist_bridge_data

## Oracle Load Rating Database

In [1]:
from sqlalchemy import create_engine, text
import json

In [2]:
def load_secrets(file_path):
    with open(file_path, 'r') as file:
        user_info = json.load(file)
    return user_info

In [3]:
from pathlib import Path

file_path = Path.home() / 'secrets.json'
dane = load_secrets(file_path)

In [4]:
# Oracle connection string using service name instead of SID
oracle_connection_string = (
    f"oracle+cx_oracle://{dane['BRR_USN']}:{dane['BRR_PASS']}@{dane['BRR_SERVER']}:{dane['BRR_PORT']}/?service_name={dane['BRR_SERVICE']}"
)

In [5]:
# Create the engine
oracle_engine = create_engine(oracle_connection_string)

In [6]:
# Establish connection
oracle_conn = oracle_engine.connect()
print("Connection successful!")

Connection successful!


In [7]:
from sqlalchemy import inspect

# Create an inspector from the engine
inspector = inspect(oracle_engine)

# Get a list of table names
tables = inspector.get_table_names()
print("Tables in the database:")
for table in tables:
    print(table)

Tables in the database:


In [8]:
query = text("""
  SELECT DISTINCT owner
  FROM all_tables
  ORDER BY owner
""")

result = oracle_conn.execute(query)
schemas = [row[0] for row in result]
print("Accessible schemas:")

for schema in schemas:
     print(schema)

Accessible schemas:
BRIDGEWARE
CTXSYS
MDSYS
ODOTREF
SYS
SYSTEM
XDB


In [9]:
schema_list = ["BRIDGEWARE", "CTXSYS", "MDSYS", "ODOTREF", "SYS", "SYSTEM", 
               "XDB"]  # Add other schemas if you want to search deeper

for schema_name in schema_list:
    print(f"\nChecking for tables in schema: {schema_name}\n")
    query = text(f"""
    SELECT table_name
    FROM all_tables
    WHERE owner = '{schema_name}'
    """)
    result = oracle_conn.execute(query)
    tables = [row[0] for row in result]
    if tables:
        print(f"Tables in schema {schema_name}:")
        for table in tables:
            print(table)
    else:
        print(f"No tables found in schema {schema_name}.")


Checking for tables in schema: BRIDGEWARE

Tables in schema BRIDGEWARE:
ABW_BRIDGE_SUB_LRFDDSN_SETTING
ABW_MA_STRUSS_ELEM_LOSS_RANGE
ABW_MCB_SEG_DECK
ABW_MATL_CONC
ABW_MATL_PS_BAR
ABW_MATL_PS_STRAND
ABW_FL_FLOORBEAM_TRAVELWAY
ABW_FLINE_MBR
ABW_MCB_TEND_PROF_DEF_DETAIL
ABW_STL_ANGLE
ABW_STL_ANGLE_CONN
ABW_STL_BEAM_ASSEMBLY
ABW_WEB_DISTRIB_LOAD
BRIDGE
COPTIONS
MULTIMEDIA
ABW_TRUSS_LINE_MBR
ABW_TRUSS_LINE_SUPPORT
ABW_TRUSSDEF_ELEMENT_CONC_LOAD
ABW_RESULTS_CONC_LS_SUMMARY
ABW_RESULTS_CONC_XSEC_PROP
ABW_LIB_MATL_TIMBER_SAWN_ITEM
ABW_SYS_LRFD_LS
ABW_SUBSDEF_MODEL_SETTING
ABW_SUPER_BR_FORCE_SUB
ABW_CONC_CURB_SIDEWALK
ABW_CONC_RAILING
ABW_CONC_RAILING_LOC
ABW_BRIDGE_DIAPHRAGM_DEF_CON
ABW_SUBSDEF_FOUND_FK
ABW_STL_BUILTUP_IBEAM_DEF
ABW_STL_LONG_STIFF
ABW_EVENT_VEHICLE_TEMPLATE
ABW_LEG_ANAL_PT_REINF_CONC
ABW_RESULTS_CRIT_LOAD_LRFD
ABW_RESULTS_DL_ACTION
ABW_RESULTS_LL_ACTION
ABW_RESULTS_LS_SUMMARY_DETAIL
ABW_RESULTS_PS_CONC_STRESS
ABW_RESULTS_RC_SERVICE_STRESS
ABW_RESULTS_SPNG_MBR_ALT_XY
ABW_RESU

In [10]:
import pandas as pd
from sqlalchemy.sql import text

# Query to fetch the entire table
query = text("SELECT * FROM BRIDGEWARE.ABW_GIRDER_MBR")

# Execute the query and fetch the data into a DataFrame
result = oracle_conn.execute(query)
df = pd.DataFrame(result.fetchall(), columns=result.keys())

In [12]:
df

,bridge_id,struct_def_id,super_struct_mbr_id,struct_def_ref_line_id,settlement_load_case_id
0,15974,1,1,5,NaN
1,15974,1,2,6,NaN
2,15974,1,3,7,NaN
3,15974,1,4,8,NaN
4,15974,1,5,9,NaN
...,...,...,...,...,...
73572,17240,1,11,15,NaN
73573,17241,1,1,5,NaN
73574,17241,1,2,6,NaN
73575,17241,1,3,7,NaN


In [13]:
def get_table(schema, table):
    query = text(f"SELECT * FROM {schema}.{table}")
    result = oracle_conn.execute(query)
    df = pd.DataFrame(result.fetchall(), columns=result.keys())
    
    return df

### BRIDGEWARE

In [14]:
from civilpy.structural.aashtoware.odot_tables import AASHTOWARE_TABLES

In [15]:
all_columns = []
column_lookup = {}
unused_tables = []

In [16]:
i = 0

for value in AASHTOWARE_TABLES.keys():
    try:
        df = get_table("BRIDGEWARE", value)
    except Exception as e:
        print(f"Database Error: {e}")
    all_columns.append(df.columns.to_list())
    column_lookup[value] = df.columns.to_list()
    if df.shape[0] == 0:
        unused_tables.append(value)
    else:
        AASHTOWARE_TABLES[value]["Used"] = True
        AASHTOWARE_TABLES[value]["Columns"] = df.columns.to_list()
        AASHTOWARE_TABLES[value]["Length"] = len(df)

    if i % 10 == 0:
        print(i)

    i += 1

0
10
20
30
40
50
60
70
80
90
100
110
120
130
140
150
160
170
180
Database Error: (cx_Oracle.DatabaseError) ORA-00942: table or view does not exist
[SQL: SELECT * FROM BRIDGEWARE.ABW_LIB_SUB_LRFD_DSG1N_SETTING]
(Background on this error at: https://sqlalche.me/e/20/4xp6)
190
200
210
220
230
240
250
260
270
280
290
300
310
320
330
340
350
360
370
380
390
400
410
420
430
440
450
460
470
480
490
500
510
520
530
540
550
560
570
580
590
600
610
620
630
640
650
660
670
680
690
700
710
720
730
740
750
760
770
780
790
800
810
820
830
840
850
860
870
880
890
900
910


In [17]:
import json
import _pickle as pickle

with open(r"C:\Users\dparks1\PycharmProjects\civilpy\src\civilpy\structural\aashtoware\odot_tables.py", 'w') as f:
    f.write("AASHTOWARE_TABLES = " + str(AASHTOWARE_TABLES))

In [26]:
sfns = get_table("BRIDGEWARE", "ABW_BRIDGE")

# Set the option to display all columns
pd.set_option('display.max_columns', None)

sfns

,bridge_id,bridge_guid,agency_code,struct_num,name,bridge_rating_ind,bridge_design_ind,bridge_management_ind,descr,elevation,x_plane_coordinate,y_plane_coordinate,prev_count_year,recent_count_year,recent_count_adtt,prev_adtt,prev_growth_rate,future_count_year,future_count_adtt,future_count_growth_rate,traffic_factor,final_design_event_id,last_modified_event_id,creation_event_id,lrfd_constant_impact_factor,lrfd_fatigue_impact_factor,impact_factor_adjustment,impact_factor_override,impact_factor_type,bridge_units_type,deleted_ind,template_ind,completely_defined_ind,last_change_timestamp,mpf_reduce_based_on_adtt_ind,traffic_adt,traffic_directional_pct,traffic_design_adtt,superstruct_filter_ind,culvert_filter_ind,custom_agency_field_one,custom_agency_field_two,custom_agency_field_three,custom_agency_field_four,custom_agency_field_five,custom_agency_field_six,custom_agency_field_seven,custom_agency_field_eight,custom_agency_field_nine,custom_agency_field_ten,fatigue_importance_factor_type,override_importance_factor_ind,importance_factor_override,sub_struct_filter_ind,exp_annual_adttsl_growth_rate,initial_adttsl,present_adttsl,limit_adttsl,featint,facility,location,yearbuilt,length,routenum,kmpost,adttotal,truckpct,latitude,longitude,district_param_id,county_param_id,funcclass_param_id,nhs_ind_param_id,adminarea_param_id,maintainer_param_id,owner_param_id,bridge_management_guid,bridge_mgmt_sync_event_id,confirm_spatial_ind,sponsoring_agency_codes,member_agency_code,sponsoring_agency_names
0,19483,339D12506E8C04FAE0630A10130D66D3,PSBD-1-25 (B42-48) - 95',B42-48 - 95' span,B42-48 - 95' span N/G,T,T,F,FAILED LOAD RATING/ DESIGN. DO NOT USE\r\nPres...,NaN,NaN,NaN,None,None,NaN,None,None,None,None,None,None,None,1204822.0,1194654.0,33.0,15.0,NaN,NaN,31401,10401,None,F,T,2025-06-16 18:19:55.857281,F,NaN,NaN,NaN,T,F,None,None,None,None,None,None,None,None,None,None,59301,F,NaN,F,NaN,NaN,NaN,NaN,None,None,None,NaN,28.95600,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,F,None,1541.0,None
1,18590,339D12506E8D04FAE0630A10130D66D3,PSBD-1-25 (CB42-48) - 90',CB42-48 - 90' span,CB42-48 - 90' span,T,T,F,"Prestressed concrete box bridge, 90' single sp...",NaN,NaN,NaN,None,None,NaN,None,None,None,None,None,None,None,1204843.0,1194654.0,33.0,15.0,NaN,NaN,31401,10401,None,F,T,2025-06-16 19:12:05.827640,F,NaN,NaN,NaN,T,F,None,None,None,None,None,None,None,None,None,None,59301,F,NaN,F,NaN,NaN,NaN,NaN,None,None,None,NaN,27.43200,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,F,None,1541.0,None
2,18873,339D12506E8E04FAE0630A10130D66D3,6104487,6104487,NOB-564-7.730,T,T,F,"Rated by RTF, PE 80163 using existing plans (1...",NaN,NaN,NaN,None,None,NaN,None,None,None,None,None,None,None,1196744.0,1196743.0,33.0,15.0,NaN,NaN,31401,10401,None,F,F,2024-11-06 17:51:59.970683,None,NaN,NaN,NaN,T,F,None,None,None,None,None,None,None,None,None,None,59301,F,NaN,F,NaN,NaN,NaN,NaN,Unnamed Stream,SR 564,9.3 Km W Of Jct SR 145,1997.0,12.92352,564,12.459541,NaN,NaN,39.67,-81.40,134.0,69.0,146.0,155.0,3.0,99.0,99.0,None,None,F,None,NaN,None
3,18874,339D12506E8F04FAE0630A10130D66D3,6300162,6300162,PAU-24-04.600,T,T,F,"Rated by RTF, PE 80163 using existing plans (2...",NaN,NaN,NaN,None,None,NaN,None,None,None,None,None,None,None,1196746.0,1196745.0,33.0,15.0,NaN,NaN,31401,10401,None,F,F,2024-11-06 17:52:01.149018,None,NaN,NaN,NaN,F,T,None,None,None,None,None,None,None,None,None,None,59301,F,NaN,F,NaN,NaN,NaN,NaN,NORTH CREEK,USR 24,US-24 Over North Creek,2009.0,6.70560,24,7.394936,NaN,NaN,41.17,-84.72,125.0,71.0,143.0,156.0,3.0,99.0,99.0,None,None,F,None,NaN,None
4,96,339D12506E9004FAE0630A10130D66D3,7004133,7004133,RIC-71-1068 R,T,T,F,"Rating by AP, from plans and standards, Septem...",NaN,0.0,0.0,None,None,0.0,None,None,None,None,None,None,None,1203852.0,5386.0,NaN,NaN,NaN,NaN,31401,10401,None,F,F,2025-05-28 19:46:36.956179,None,NaN,NaN,NaN,T,F,None,None,None,None,None,None,None,None,None,None,59301,F,NaN,T,NaN,NaN,NaN,NaN,SR 13,IR 71,3.56 mi N of SR

In [65]:
bridge_sfn2 = '2534231'

single_bridge = sfns[sfns['agency_code'] == bridge_sfn2]
bridge_index = single_bridge['bridge_id'].tolist()[0]
bridge_index

361

In [66]:
from IPython.display import display, HTML

for key, value in column_lookup.items():
    if 'bridge_id' in value:
        df = get_table("BRIDGEWARE", key)
        df = df[df['bridge_id'] == bridge_index]
        if len(df) > 0:
            print(f"{key}: ")
            display(HTML(df.to_html()))

ABW_MATL_CONC: 


,bridge_id,conc_id,name,descr,si_or_us_type,comp_strength_28,initial_comp_strength,thermal_exp_coeff,density_dl,density_e,std_mod_of_elast,poisson_ratio,composition_type,shear_factor,std_initial_mod_of_elast,last_change_timestamp,shrinkage_coef,lrfd_mod_of_elast,lrfd_initial_mod_of_elast,splitting_tensile_strength,lrfd_max_aggregate_size,std_mod_of_rupture,lrfd_mod_of_rupture
416,361,1,5.5 ksi Beam Concrete,None,10401,37.921163,27.579028,0.000011,2402.809922,2402.809922,30999.246507,0.2,34601,1.0,26436.246247,2008-11-25 20:08:23,None,30999.246507,26436.246247,None,NaN,NaN,3.880713


ABW_MATL_PS_STRAND: 


,bridge_id,matl_ps_strand_id,name,descr,si_or_us_type,strand_type,strand_diameter,strand_area,weight_per_length,ultimate_tensile_strength,yield_strength,mod_of_elast,transfer_length_lrfd,transfer_length_std,epoxy_coated_ind,last_change_timestamp
27,361,1,"1/2"" (7W-270) SR","Stress relieved 1/2""/Seven Wire/fpu = 270",10401,34302,12.7,98.70948,0.773858,1861.58439,1582.346732,196500.5745,762.0,635.0,F,2008-11-25 20:08:23


ABW_SUPER_LOAD_SCENARIO: 


...

,bridge_id,struct_def_id,super_struct_mbr_id,super_struct_spng_mbr_alt_id,name,descr,spng_mbr_def_id,impact_factor_override,import_event_id,impact_factor_adjustment,last_modified_event_id,creation_event_id,lrfd_constant_impact_factor,lrfd_fatigue_impact_factor,impact_factor_type,units_type,asr_analysis_module_guid,lfr_analysis_module_guid,lrfd_analysis_module_guid,default_analysis_method_type,comment_id,lfd_single_ll_fact_moment,lfd_single_ll_fact_shear,lfd_single_ll_fact_shear_supp,lfd_single_ll_fact_deflection,lfd_multi_ll_fact_moment,lfd_multi_ll_fact_shear,lfd_multi_ll_fact_shear_supp,lfd_multi_ll_fact_deflection,haunch_embedded_flng_ind,deck_type,validation_status_type,import_status_type,override_lfr_factor_id,override_lrfd_factor_id,asr_inv_rebar_factor,asr_inv_concrete_factor,asr_inv_structural_stl_factor,asr_inv_stirrup_factor,asr_inv_bearing_stiff_factor,asr_opr_rebar_factor,asr_opr_concrete_factor,asr_opr_structural_stl_factor,asr_opr_stirrup_factor,asr_opr_bearing_stiff_factor,asr_post_rebar_factor,asr_post_concrete_factor,asr_post_structural_stl_factor,asr_post_stirrup_factor,asr_post_bearing_stiff_factor,asr_sl_rebar_factor,asr_sl_concrete_factor,asr_sl_structural_stl_factor,asr_sl_stirrup_factor,asr_sl_bearing_stiff_factor,computed_deck_schedule_ind,computed_reinf_schedule_ind,modular_ratio_sustained_factor,haunch_dim_type,deck_curing_method_type,asr_inv_ps_conc_comp_factor,asr_inv_ps_conc_tension_factor,asr_inv_ps_mom_capacity_factor,asr_opr_ps_conc_comp_factor,asr_opr_ps_conc_tension_factor,asr_opr_ps_mom_capacity_factor,asr_pst_ps_conc_comp_factor,asr_pst_ps_conc_tension_factor,asr_pst_ps_mom_capacity_factor,asr_sl_ps_conc_comp_factor,asr_sl_ps_conc_tension_factor,asr_sl_ps_mom_capacity_factor,deck_interface_type,deck_cohesion_factor,deck_friction_factor,deck_interface_width,default_int_width_to_beam_ind,service_life,time_composite,time_analysis,time_continuous,deck_drying_time,deck_differential_shrink_ind,beam_projection_start,beam_projection_end,asr_opr_timber_adj_allw_stress,override_assigned_stringer_ind,last_change_timestamp,fb_lfd_ll_fact_shear_moment,fb_lfd_ll_fact_deflection,fb_lrfd_ll_fact_shear_moment,fb_lrfd_ll_fact_deflection,lrfr_analysis_module_guid,nsg_analysis_module_guid,override_lrfr_factor_id,dist_factor_input_method_type,read_only_ind,inv_lfd_single_ll_fact_moment,inv_lfd_single_ll_fact_shear,inv_lfd_single_ll_fact_shr_sup,inv_lfd_single_ll_fact_defl,inv_lfd_multi_ll_fact_moment,inv_lfd_multi_ll_fact_shear,inv_lfd_multi_ll_fact_shr_sup,inv_lfd_multi_ll_fact_defl,opr_lfd_single_ll_fact_moment,opr_lfd_single_ll_fact_shear,opr_lfd_single_ll_fact_shr_sup,opr_lfd_single_ll_fact_defl,opr_lfd_multi_ll_fact_moment,opr_lfd_multi_ll_fact_shear,opr_lfd_multi_ll_fact_shr_sup,opr_lfd_multi_ll_fact_defl,lrfd_lldist_suff_conn_unit_ind,conc_str_fraction_int_shear_k1,limit_int_shear_resistance_k2,override_asr_factor_id,asr_spec_edition_choice_type,lfr_spec_edition_choice_type,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,std_dist_fact_perm_load_ind,lrfd_dist_fact_perm_load_ind,slab_edge_beam_ind,analysis_strip_type,analysis_strip_user_def_val,deck_load_case_id,deck_loadcase_engineassign_ind,ll_lrfd_conn_act_as_unit_ind,lrfd_dist_fact_in_method_type,super_struct_spng_mbr_alt_type,std_dist_fact_computed_date,lrfd_dist_fact_computed_date,shear_conn_input_method_type
77,361,1,1,1,Precast Box Alternative,None,1,0.0,NaN,0.0,NaN,11993.0,33.0,15.0,31401,10401,75BA513B-0C0A-4E92-8F5D-FE4233875977,DAF6E1A1-2B96-49E9-A99D-9625BFB527CC,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,33902,None,0.702001,0.702001,1.0,0.222222,0.715052,0.715052,1.0,0.6,None,32701.0,NaN,NaN,NaN,NaN,NaN,None,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,T,F,2.0,32201.0,NaN,None,NaN,None,None,NaN,None,None,None,None,None,None,None,35101.0,1.034214,NaN,NaN,T,75.0,NaN,NaN,NaN,NaN,F,NaN,NaN,NaN,None,2008-11-25 

ABW_BRIDGE_WIND_EFFECT: 


,bridge_id,environmental_param_id,wind_effect_id,skew_angle,normal_component_on_ll,parallel_component_on_ll,base_press_arch_col_truss_lat,base_press_arch_col_truss_long,base_press_girder_lat,base_press_girder_long,gust_normal_component_on_ll,gust_parallel_component_on_ll,gust_skcoeff_archcoltruss_lat,gust_skcoeff_archcoltruss_long,gust_skcoeff_girder_lat,gust_skcoeff_girder_long
1494,361,1,1,0.0,1.460000,0.000000,0.003600,0.000000,0.002400,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
1495,361,1,2,15.0,1.280000,0.180000,0.003400,0.000600,0.002100,0.000300,NaN,NaN,NaN,NaN,NaN,NaN
1496,361,1,3,30.0,1.200000,0.350000,0.003100,0.001300,0.002000,0.000600,NaN,NaN,NaN,NaN,NaN,NaN
1497,361,1,4,45.0,0.960000,0.470000,0.002300,0.002000,0.001600,0.000800,NaN,NaN,NaN,NaN,NaN,NaN
1498,361,1,5,60.0,0.500000,0.550000,0.001100,0.002400,0.000800,0.000900,NaN,NaN,NaN,NaN,NaN,NaN
1499,361,2,1,0.0,1.459392,0.000000,0.003591,0.000000,0.002394,0.000000,1.459392,0.000000,1.000,0.000,1.00,0.00
1500,361,2,2,15.0,1.284265,0.175127,0.003352,0.000575,0.002107,0.000287,1.284265,0.175127,0.933,0.160,0.88,0.12
1501,361,2,3,30.0,1.196701,0.350254,0.003112,0.001341,0.001963,0.000575,1.196701,0.350254,0.867,0.373,0.82,0.24
1502,361,2,4,45.0,0.963198,0.467005,0.002250,0.001963,0.001580,0.000766,0.963198,0.467005,0.627,0.547,0.66,0.32
24240,361,2,5,60.0,0.496193,0.554569,0.001149,0.002394,0.000814,0.000910,0.496193,0.554569,0.320,0.667,0.34,0.38


ABW_PS_PRECAST_BOX_BEAM_DEF: 


,bridge_id,struct_def_id,spng_mbr_def_id
78,361,1,1
79,361,1,2
80,361,1,3


In [67]:
bridge_index

361

In [63]:
single_bridge

,bridge_id,bridge_guid,agency_code,struct_num,name,bridge_rating_ind,bridge_design_ind,bridge_management_ind,descr,elevation,x_plane_coordinate,y_plane_coordinate,prev_count_year,recent_count_year,recent_count_adtt,prev_adtt,prev_growth_rate,future_count_year,future_count_adtt,future_count_growth_rate,traffic_factor,final_design_event_id,last_modified_event_id,creation_event_id,lrfd_constant_impact_factor,lrfd_fatigue_impact_factor,impact_factor_adjustment,impact_factor_override,impact_factor_type,bridge_units_type,deleted_ind,template_ind,completely_defined_ind,last_change_timestamp,mpf_reduce_based_on_adtt_ind,traffic_adt,traffic_directional_pct,traffic_design_adtt,superstruct_filter_ind,culvert_filter_ind,custom_agency_field_one,custom_agency_field_two,custom_agency_field_three,custom_agency_field_four,custom_agency_field_five,custom_agency_field_six,custom_agency_field_seven,custom_agency_field_eight,custom_agency_field_nine,custom_agency_field_ten,fatigue_importance_factor_type,override_importance_factor_ind,importance_factor_override,sub_struct_filter_ind,exp_annual_adttsl_growth_rate,initial_adttsl,present_adttsl,limit_adttsl,featint,facility,location,yearbuilt,length,routenum,kmpost,adttotal,truckpct,latitude,longitude,district_param_id,county_param_id,funcclass_param_id,nhs_ind_param_id,adminarea_param_id,maintainer_param_id,owner_param_id,bridge_management_guid,bridge_mgmt_sync_event_id,confirm_spatial_ind,sponsoring_agency_codes,member_agency_code,sponsoring_agency_names
263,361,339D12506F8704FAE0630A10130D66D3,2534231,2534231,MAD-C0376-0087,T,T,F,None,NaN,0.0,0.0,None,None,0.0,None,None,None,None,None,None,None,154636.0,11979.0,33.0,15.0,NaN,NaN,31401,10401,None,F,F,2015-03-09 18:22:41,None,NaN,NaN,NaN,T,F,None,None,None,None,None,None,None,None,None,None,59301,None,NaN,T,NaN,NaN,NaN,NaN,Georges Creek (WinchPike,Winchester Pike,None,1978.0,23.7744,C0376,1.400129,NaN,NaN,NaN,NaN,130.0,33.0,152.0,NaN,NaN,102.0,102.0,None,None,None,None,NaN,None


In [ ]:
print(df.columns.to_list())
print(df.shape)

## Appendix